In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors,AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from matplotlib.colors import ListedColormap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
import rdkit.Chem.Descriptors3D
from morfeus import xtb, read_xyz
import os

In [2]:
file_path = './MixtureSolDB_with_features.csv'

df = pd.read_csv(file_path)

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# группы — по веществу
groups = df['SMILES_Solute']

# --- 1. train (70%) / temp (30%) ---
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

# --- 2. temp → val (15%) / test (15%) ---
groups_temp = temp_df['SMILES_Solute']

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=groups_temp))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

# проверка
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# контроль утечки
print("\nUnique solutes:")
print("Train:", train_df['SMILES_Solute'].nunique())
print("Val:", val_df['SMILES_Solute'].nunique())
print("Test:", test_df['SMILES_Solute'].nunique())

print("\nIntersection check:")
print("Train ∩ Val:", len(set(train_df['SMILES_Solute']) & set(val_df['SMILES_Solute'])))
print("Train ∩ Test:", len(set(train_df['SMILES_Solute']) & set(test_df['SMILES_Solute'])))
print("Val ∩ Test:", len(set(val_df['SMILES_Solute']) & set(test_df['SMILES_Solute'])))

Train: (74908, 127)
Val: (23583, 127)
Test: (18811, 127)

Unique solutes:
Train: 389
Val: 84
Test: 84

Intersection check:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [4]:
from sklearn.impute import SimpleImputer

target_col = 'LogS(mole_fraction)'

drop_cols = [
    target_col,
    'Solubility(mole_fraction)',
    'Compound_Name',
    'Solvent1',
    'Solvent2',
    'SMILES_Solute',
    'SMILES_Solvent1',
    'SMILES_Solvent2'
]

feature_cols = [c for c in df.columns if c not in drop_cols]

print("Number of features:", len(feature_cols))
print(feature_cols[:30])

Number of features: 119
['Temperature_K', 'x1', 'x2', 'IsPureSolventEndpoint', 'S1_pure_solubility', 'S1_pure_logS', 'S1_temp_diff', 'S2_pure_solubility', 'S2_pure_logS', 'S2_temp_diff', 'solute_MolWt', 'solute_MolLogP', 'solute_TPSA', 'solute_NumHDonors', 'solute_NumHAcceptors', 'solute_NumRotatableBonds', 'solute_RingCount', 'solute_NumAromaticRings', 'solute_FractionCSP3', 'solute_HeavyAtomCount', 'solute_BertzCT', 'solute_BalabanJ', 'solute_MaxPartialCharge', 'solute_MinPartialCharge', 'solv1_MolWt', 'solv1_MolLogP', 'solv1_TPSA', 'solv1_NumHDonors', 'solv1_NumHAcceptors', 'solv1_NumRotatableBonds']


In [5]:
X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

y_train = train_df[target_col].copy()
y_val   = val_df[target_col].copy()
y_test  = test_df[target_col].copy()

# на всякий случай оставляем только числовые признаки
X_train = X_train.select_dtypes(include=[np.number])
X_val   = X_val[X_train.columns]
X_test  = X_test[X_train.columns]

print(X_train.shape, X_val.shape, X_test.shape)

(74908, 118) (23583, 118) (18811, 118)


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    'LR': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ]),
    
    'DT': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', DecisionTreeRegressor(
            max_depth=20,
            min_samples_leaf=5,
            random_state=42
        ))
    ]),
    
    'RF': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=42
        ))
    ]),

    #'kNN': Pipeline([
    #    ('imputer', SimpleImputer(strategy='median')),
    #    ('scaler', StandardScaler()),
    #    ('model', KNeighborsRegressor(n_neighbors=7, weights='distance'))
    #]),
}

In [30]:
from tqdm import tqdm

results = []

for name, model in tqdm(models.items(), desc="Training models"):
    print(f"\n===== {name} =====")
    
    model.fit(X_train, y_train)
    
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)
    
    rmse_val = mean_squared_error(y_val, pred_val) ** 0.5
    r2_val = r2_score(y_val, pred_val)
    
    rmse_test = mean_squared_error(y_test, pred_test) ** 0.5
    r2_test = r2_score(y_test, pred_test)
    
    print(f"VAL  | RMSE: {rmse_val:.4f} | R2: {r2_val:.4f}")
    print(f"TEST | RMSE: {rmse_test:.4f} | R2: {r2_test:.4f}")
    
    results.append({
        'Model': name,
        'RMSE_val': rmse_val,
        'R2_val': r2_val,
        'RMSE_test': rmse_test,
        'R2_test': r2_test
    })

results_df = pd.DataFrame(results).sort_values('RMSE_val').reset_index(drop=True)

print("\n===== FINAL RESULTS =====")
display(results_df)

Training models:   0%|                                    | 0/3 [00:00<?, ?it/s]


===== LR =====


Training models:  33%|█████████▎                  | 1/3 [00:00<00:00,  2.03it/s]

VAL  | RMSE: 0.3121 | R2: 0.9287
TEST | RMSE: 0.2757 | R2: 0.9520

===== DT =====


Training models:  67%|██████████████████▋         | 2/3 [00:02<00:01,  1.67s/it]

VAL  | RMSE: 0.3403 | R2: 0.9152
TEST | RMSE: 0.3403 | R2: 0.9270

===== RF =====


Training models: 100%|████████████████████████████| 3/3 [01:07<00:00, 22.45s/it]

VAL  | RMSE: 0.2525 | R2: 0.9533
TEST | RMSE: 0.2530 | R2: 0.9596

===== FINAL RESULTS =====


,Model,RMSE_val,R2_val,RMSE_test,R2_test
0,RF,0.252508,0.953323,0.253027,0.959619
1,LR,0.312111,0.928687,0.275747,0.952042
2,DT,0.340292,0.915228,0.340258,0.926977


In [10]:
# достаём обученную модель
lr_model = models['LR']
lr = lr_model.named_steps['model']

feature_names = X_train.columns

coefs = pd.DataFrame({
    'feature': feature_names,
    'coef': lr.coef_
})

coefs['abs_coef'] = coefs['coef'].abs()
coefs = coefs.sort_values('abs_coef', ascending=False)

print("Top features (LR):")
display(coefs.head(20))

Top features (LR):


,feature,coef,abs_coef
51,mix_logS_linear,0.768109,0.768109
59,x2_logS2,0.486287,0.486287
58,x1_logS1,0.311896,0.311896
61,x1x2_abs_logS_diff,0.205727,0.205727
106,mix_avg_BalabanJ,-0.169932,0.169932
48,solv2_BalabanJ,0.137429,0.137429
60,x1x2_logS_diff,0.108587,0.108587
34,solv1_BalabanJ,0.107660,0.107660
104,mix_abs_diff_BertzCT,0.106757,0.106757
110,mix_avg_MaxPartialCharge,0.096945,0.096945


In [11]:
rf_model = models['RF']
rf = rf_model.named_steps['model']

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top features (RF):")
display(importances.head(20))

Top features (RF):


,feature,importance
51,mix_logS_linear,0.825961
59,x2_logS2,0.034881
60,x1x2_logS_diff,0.030748
3,S1_pure_solubility,0.016231
61,x1x2_abs_logS_diff,0.012595
4,S1_pure_logS,0.009774
117,mix_x1x2_abs_diff_MinPartialCharge,0.006004
109,mix_x1x2_abs_diff_BalabanJ,0.004895
65,mix_x1x2_abs_diff_MolWt,0.003077
97,mix_x1x2_abs_diff_FractionCSP3,0.003004


In [22]:
logS_related = [
    'mix_logS_linear',
    'S1_pure_solubility',
    'S2_pure_solubility',
    'S1_pure_logS',
    'S2_pure_logS',
    'x1_logS1',
    'x2_logS2',
    'pure_logS_diff',
    'pure_logS_abs_diff',
    'x1x2_logS_diff',
    'x1x2_abs_logS_diff'
]

def evaluate_without_logS_features():
    cols = [c for c in feature_cols if c not in logS_related]
    
    X_tr = train_df[cols].select_dtypes(include=[np.number])
    X_v  = val_df[cols][X_tr.columns]
    X_te = test_df[cols][X_tr.columns]
    
    model = models['RF']
    model.fit(X_tr, y_train)
    
    pred_val = model.predict(X_v)
    pred_test = model.predict(X_te)
    
    rmse_val = mean_squared_error(y_val, pred_val) ** 0.5
    r2_val = r2_score(y_val, pred_val)
    
    rmse_test = mean_squared_error(y_test, pred_test) ** 0.5
    r2_test = r2_score(y_test, pred_test)
    
    print("\nWithout ANY pure solubility info:")
    print(f"VAL  | RMSE: {rmse_val:.4f} | R2: {r2_val:.4f}")
    print(f"TEST | RMSE: {rmse_test:.4f} | R2: {r2_test:.4f}")

In [23]:
evaluate_without_logS_features()


Without ANY pure solubility info:
VAL  | RMSE: 1.5587 | R2: -0.7786
TEST | RMSE: 1.3539 | R2: -0.1562


In [24]:
cols = [c for c in feature_cols if c not in logS_related]

X_tr = train_df[cols].select_dtypes(include=[np.number])
X_v  = val_df[cols][X_tr.columns]
X_te = test_df[cols][X_tr.columns]

y_tr = y_train
y_v  = y_val
y_te = y_test

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score

lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_tr, y_tr)

pred_val = lr_pipeline.predict(X_v)
pred_test = lr_pipeline.predict(X_te)

rmse_val = mean_squared_error(y_v, pred_val) ** 0.5
r2_val = r2_score(y_v, pred_val)

rmse_test = mean_squared_error(y_te, pred_test) ** 0.5
r2_test = r2_score(y_te, pred_test)

print("LR without pure solubility:")
print(f"VAL  | RMSE: {rmse_val:.4f} | R2: {r2_val:.4f}")
print(f"TEST | RMSE: {rmse_test:.4f} | R2: {r2_test:.4f}")

LR without pure solubility:
VAL  | RMSE: 0.9685 | R2: 0.3133
TEST | RMSE: 1.0919 | R2: 0.2480


In [26]:
lr = lr_pipeline.named_steps['model']

coefs = pd.DataFrame({
    'feature': X_tr.columns,
    'coef': lr.coef_
})

coefs['abs_coef'] = coefs['coef'].abs()
coefs = coefs.sort_values('abs_coef', ascending=False)

print("\nTop features (LR, no pure solubility):")
display(coefs.head(20))


Top features (LR, no pure solubility):


,feature,coef,abs_coef
42,solv2_HeavyAtomCount,-1.037611,1.037611
14,solute_HeavyAtomCount,1.010549,1.010549
15,solute_BertzCT,-0.939991,0.939991
55,mix_avg_MolLogP,-0.804295,0.804295
88,mix_diff_HeavyAtomCount,0.653945,0.653945
83,mix_avg_FractionCSP3,0.571859,0.571859
34,solv2_MolLogP,0.563928,0.563928
44,solv2_BalabanJ,0.439752,0.439752
95,mix_avg_BalabanJ,0.411097,0.411097
93,mix_abs_diff_BertzCT,0.408821,0.408821


In [28]:
df.shape

(117302, 127)